In [22]:
import os
import glob
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from pathlib import Path, PurePosixPath
from collections import defaultdict
from glob import iglob
import re

In [5]:
df_1_aug_data = pd.read_json("/fastdata3/vlm_data_parquets/1_Aug_final_data_for_pretraining_cleaned.json", lines=True)

In [6]:
df_1_aug_data

,image,messages
0,/raid3/cxr_data/training/images/bodyscans.1.2....,"[{'from': 'human', 'value': 'Generate the bbox..."
1,/raid1/akshay/mimic_data/physionet.org/files/m...,"[{'from': 'human', 'value': 'Give a detailed r..."
2,/raid3/cxr_data/training/images/fiveceu.2d46c6...,"[{'from': 'human', 'value': 'Generate the bbox..."
3,/raid3/cxr_data/training/images/fiveceu.656009...,"[{'from': 'human', 'value': 'Generate the bbox..."
4,/raid1/fake_data/fake_imgs/lung_20240821_02590...,"[{'from': 'human', 'value': 'Do the detailed q..."
...,...,...
863093,/data_nas5/akhila/hollow_nodules/hollow_nodule...,"[{'from': 'human', 'value': 'Do the detailed a..."
863094,/raid/segmed_data/segmed_images/1.3.6.1.4.1.55...,"[{'from': 'human', 'value': 'Generate the Answ..."
863095,/raid1/fake_data/fake_imgs/lung_20240821_08382...,"[{'from': 'human', 'value': 'Do the detailed q..."
863096,/raid3/cxr_data/training/images/image_700x700_...,"[{'from': 'human', 'value': 'Generate the bbox..."


In [8]:
unique_base_dirs = sorted(df_1_aug_data['image'].map(os.path.dirname).unique())
print(f"Unique base directories: {len(unique_base_dirs)}")
# unique_base_dirs


Unique base directories: 40845


In [17]:
df_1_aug_data["image"].iloc[863096]


'/raid3/cxr_data/training/images/image_700x700_chex.train.patient35620.study10.view1_frontal.png'

In [15]:
unique_base_dirs[9]

'/raid/opensource_datasets/PNG/pngs/PNG/train/patient00098/study2'

In [111]:
BASENAME_DIRS = [
    "/localstorage/cxr_data/internal_data/training/images/images",
    "/localstorage/ph_data/ph_images",
    "/fastdata3/cxr_data/opensource_datasets/mimic/images",
    "/fastdata3/cxr_data/opensource_datasets/padchest",
    "/fastdata3/cxr_data/opensource_datasets/chexpert/pngs/PNG/train",
    "/fastdata3/cxr_data/segmed_data/images",
    "/localstorage/cxr_data/fake_data/",
    "/localstorage/prod_data/prod_images/",
]

In [112]:
import os
import re
import math
import pandas as pd
from pathlib import Path, PurePosixPath
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

EXTS = {".png", ".jpg", ".jpeg"}
MAX_TAIL_PARTS = 6
AMBIG = "__AMBIGUOUS__"
N_WORKERS = min(16, os.cpu_count() or 4)

SIZE_PREFIX_RE = re.compile(r"^(?:image_)?\d+(?:x|_)\d+_", re.IGNORECASE)


def strip_size_prefix(filename):
    return SIZE_PREFIX_RE.sub("", filename)


def add_unique(index, n, key, value):
    old = index[n].get(key)
    if old is None:
        index[n][key] = value
    elif old != value:
        index[n][key] = AMBIG


def scan_one_base(base):
    base_path = Path(base)
    tail_index = defaultdict(dict)
    stem_tail_index = defaultdict(dict)
    n_files = 0

    for root, _, files in os.walk(base):
        for fname in files:
            if os.path.splitext(fname)[1].lower() not in EXTS:
                continue

            n_files += 1
            full_path = os.path.join(root, fname)
            p = Path(full_path)

            try:
                rel_parts = p.relative_to(base_path).parts
            except ValueError:
                continue

            max_n = min(MAX_TAIL_PARTS, len(rel_parts))

            for n in range(1, max_n + 1):
                tail = "/".join(rel_parts[-n:]).lower()
                add_unique(tail_index, n, tail, full_path)

                stem_parts = list(rel_parts[-n:])
                stem_parts[-1] = os.path.splitext(stem_parts[-1])[0]
                stem_tail = "/".join(stem_parts).lower()
                add_unique(stem_tail_index, n, stem_tail, full_path)

    return tail_index, stem_tail_index, n_files


def merge_index(dst, src):
    for n, src_dict in src.items():
        dst_dict = dst[n]
        for key, value in src_dict.items():
            old = dst_dict.get(key)
            if old is None:
                dst_dict[key] = value
            elif old != value:
                dst_dict[key] = AMBIG


tail_index = defaultdict(dict)
stem_tail_index = defaultdict(dict)
total_files = 0

with ProcessPoolExecutor(max_workers=min(len(BASENAME_DIRS), N_WORKERS)) as ex:
    futures = [ex.submit(scan_one_base, base) for base in BASENAME_DIRS]

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Scanning base dirs"):
        partial_tail, partial_stem_tail, n_files = fut.result()
        merge_index(tail_index, partial_tail)
        merge_index(stem_tail_index, partial_stem_tail)
        total_files += n_files

print("indexed files:", total_files)

Scanning base dirs:   0%|          | 0/8 [00:00<?, ?it/s]

indexed files: 6051657


In [113]:
print("indexed files:", total_files)

indexed files: 6051657


In [33]:
_GLOBAL_TAIL_INDEX = None
_GLOBAL_STEM_TAIL_INDEX = None


def init_worker(tail_idx, stem_tail_idx):
    global _GLOBAL_TAIL_INDEX, _GLOBAL_STEM_TAIL_INDEX
    _GLOBAL_TAIL_INDEX = tail_idx
    _GLOBAL_STEM_TAIL_INDEX = stem_tail_idx


def candidate_keys(old_path):
    if pd.isna(old_path):
        return

    parts = [
        p for p in PurePosixPath(str(old_path)).parts
        if p not in ("", "/")
    ]

    if not parts:
        return

    filename = parts[-1]
    filenames = [filename]

    stripped = strip_size_prefix(filename)
    if stripped != filename:
        filenames.append(stripped)

    for fname in filenames:
        fixed_parts = parts[:-1] + [fname]
        max_n = min(MAX_TAIL_PARTS, len(fixed_parts))

        for n in range(max_n, 0, -1):
            yield _GLOBAL_TAIL_INDEX, n, "/".join(fixed_parts[-n:]).lower()

            stem_parts = list(fixed_parts[-n:])
            stem_parts[-1] = os.path.splitext(stem_parts[-1])[0]
            yield _GLOBAL_STEM_TAIL_INDEX, n, "/".join(stem_parts).lower()


def find_new_image_path(old_path):
    for index, n, key in candidate_keys(old_path) or []:
        hit = index[n].get(key)
        if hit and hit != AMBIG:
            return hit

    return pd.NA


def map_chunk(chunk):
    return chunk.map(find_new_image_path)


image_series = df_1_aug_data["image"]
chunk_size = math.ceil(len(image_series) / N_WORKERS)
chunks = [
    image_series.iloc[i:i + chunk_size]
    for i in range(0, len(image_series), chunk_size)
]

results = []

with ProcessPoolExecutor(
    max_workers=N_WORKERS,
    initializer=init_worker,
    initargs=(tail_index, stem_tail_index),
) as ex:
    futures = [ex.submit(map_chunk, chunk) for chunk in chunks]

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Mapping dataframe"):
        results.append(fut.result())

new_image = pd.concat(results).sort_index()
df_1_aug_data["new_image"] = new_image

print("rows:", len(df_1_aug_data))
print("mapped:", df_1_aug_data["new_image"].notna().sum())
print("missing:", df_1_aug_data["new_image"].isna().sum())

Mapping dataframe:   0%|          | 0/16 [00:00<?, ?it/s]

rows: 863098
mapped: 833460
missing: 29638


In [35]:
print(df_1_aug_data["new_image"].iloc[10])

/localstorage/cxr_data/internal_data/training/images/images/fiveceu.955f22ca-36e943e9-a77b3fed-50d9d769-ab38372d.png


In [34]:
df_1_aug_data

,image,messages,new_image
0,/raid3/cxr_data/training/images/bodyscans.1.2....,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
1,/raid1/akshay/mimic_data/physionet.org/files/m...,"[{'from': 'human', 'value': 'Give a detailed r...",/fastdata3/cxr_data/opensource_datasets/mimic/...
2,/raid3/cxr_data/training/images/fiveceu.2d46c6...,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
3,/raid3/cxr_data/training/images/fiveceu.656009...,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
4,/raid1/fake_data/fake_imgs/lung_20240821_02590...,"[{'from': 'human', 'value': 'Do the detailed q...",/localstorage/cxr_data/fake_data/lung_20240821...
...,...,...,...
863093,/data_nas5/akhila/hollow_nodules/hollow_nodule...,"[{'from': 'human', 'value': 'Do the detailed a...",<NA>
863094,/raid/segmed_data/segmed_images/1.3.6.1.4.1.55...,"[{'from': 'human', 'value': 'Generate the Answ...",/fastdata3/cxr_data/segmed_data/images/1.3.6.1...
863095,/raid1/fake_data/fake_imgs/lung_20240821_08382...,"[{'from': 'human', 'value': 'Do the detailed q...",/localstorage/cxr_data/fake_data/lung_20240821...
863096,/raid3/cxr_data/training/images/image_700x700_...,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...


In [36]:
import random


def basenames_match(old_path, new_path):
    if pd.isna(new_path):
        return False
    old_base = os.path.basename(old_path)
    new_base = os.path.basename(new_path)
    if old_base == new_base:
        return True
    return strip_size_prefix(old_base) == new_base


print(df_1_aug_data["new_image"].iloc[10])
print("old basename:", os.path.basename(df_1_aug_data["image"].iloc[10]))
print("new basename:", os.path.basename(df_1_aug_data["new_image"].iloc[10]))

mapped = df_1_aug_data[df_1_aug_data["new_image"].notna()]
sample_idxs = random.sample(mapped.index.tolist(), k=min(20, len(mapped)))

print("\nRandom filename verification:")
for idx in sample_idxs:
    old_path = df_1_aug_data.at[idx, "image"]
    new_path = df_1_aug_data.at[idx, "new_image"]
    old_base = os.path.basename(old_path)
    new_base = os.path.basename(new_path)
    match = basenames_match(old_path, new_path)
    print(f"idx={idx} match={match}")
    print(f"  old: {old_base}")
    print(f"  new: {new_base}")
    if strip_size_prefix(old_base) != old_base:
        print(f"  stripped old: {strip_size_prefix(old_base)}")
    print()

matches = mapped.apply(lambda row: basenames_match(row["image"], row["new_image"]), axis=1)
print(f"Mapped rows with matching filename: {matches.sum()} / {len(mapped)}")
print(f"Mismatches: {(~matches).sum()}")

if (~matches).any():
    print("\nMismatch examples:")
    for idx in mapped.index[~matches][:5]:
        print(f"idx={idx}")
        print(f"  old: {df_1_aug_data.at[idx, 'image']}")
        print(f"  new: {df_1_aug_data.at[idx, 'new_image']}")

/localstorage/cxr_data/internal_data/training/images/images/fiveceu.955f22ca-36e943e9-a77b3fed-50d9d769-ab38372d.png
old basename: fiveceu.955f22ca-36e943e9-a77b3fed-50d9d769-ab38372d.png
new basename: fiveceu.955f22ca-36e943e9-a77b3fed-50d9d769-ab38372d.png

Random filename verification:
idx=368960 match=True
  old: view1_frontal.png
  new: view1_frontal.png

idx=670264 match=True
  old: image_812x812_medall.1.2.840.10008.1.890041.1599922007.309318695.2050962742.png
  new: medall.1.2.840.10008.1.890041.1599922007.309318695.2050962742.png
  stripped old: medall.1.2.840.10008.1.890041.1599922007.309318695.2050962742.png

idx=594110 match=True
  old: chex.train.patient03571.study6.view1_frontal.png
  new: chex.train.patient03571.study6.view1_frontal.png

idx=783442 match=True
  old: fiveceu.6a67c655-66928715-1370d26e-ca50d4ba-98bcebba.png
  new: fiveceu.6a67c655-66928715-1370d26e-ca50d4ba-98bcebba.png

idx=348176 match=True
  old: 1.3.6.1.4.1.55648.010957224166567675879759731385702522945

In [37]:
df_1_aug_data = df_1_aug_data.drop(columns=["image"]).rename(columns={"new_image": "image"})
df_1_aug_data

,messages,image
0,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
1,"[{'from': 'human', 'value': 'Give a detailed r...",/fastdata3/cxr_data/opensource_datasets/mimic/...
2,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
3,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
4,"[{'from': 'human', 'value': 'Do the detailed q...",/localstorage/cxr_data/fake_data/lung_20240821...
...,...,...
863093,"[{'from': 'human', 'value': 'Do the detailed a...",<NA>
863094,"[{'from': 'human', 'value': 'Generate the Answ...",/fastdata3/cxr_data/segmed_data/images/1.3.6.1...
863095,"[{'from': 'human', 'value': 'Do the detailed q...",/localstorage/cxr_data/fake_data/lung_20240821...
863096,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...


In [38]:
df_1_aug_data

,messages,image
0,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
1,"[{'from': 'human', 'value': 'Give a detailed r...",/fastdata3/cxr_data/opensource_datasets/mimic/...
2,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
3,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...
4,"[{'from': 'human', 'value': 'Do the detailed q...",/localstorage/cxr_data/fake_data/lung_20240821...
...,...,...
863093,"[{'from': 'human', 'value': 'Do the detailed a...",<NA>
863094,"[{'from': 'human', 'value': 'Generate the Answ...",/fastdata3/cxr_data/segmed_data/images/1.3.6.1...
863095,"[{'from': 'human', 'value': 'Do the detailed q...",/localstorage/cxr_data/fake_data/lung_20240821...
863096,"[{'from': 'human', 'value': 'Generate the bbox...",/localstorage/cxr_data/internal_data/training/...


In [39]:
df_1_aug_data.dropna(subset=["image"], inplace=True)

In [41]:
df_1_aug_data.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/1_Aug_final_data_for_pretraining_cleaned_neysa.json", lines=True, orient="records")

In [42]:
df_1_aug_data["image"].map(os.path.exists).all()

np.True_

# dataset 2

In [43]:
df_2_= pd.read_json("/fastdata3/vlm_data_parquets/15_July_final_segmed_chexpert_mimic_vlm_conv_gemini_response.json", lines=True)

In [44]:
df_2_

,images,messages
0,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate a detail..."
1,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate a medica..."
2,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate a medica..."
3,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Impr..."
4,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Find..."
...,...,...
1055746,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Summ..."
1055747,[/raid1/akshay/mimic_data/physionet.org/files/...,"[{'from': 'human', 'value': 'Generate the Impr..."
1055748,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Summ..."
1055749,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Prom..."


In [46]:
from tqdm.auto import tqdm
import pandas as pd
import os
import re
from pathlib import PurePosixPath

tqdm.pandas()

SIZE_PREFIX_RE = re.compile(r"^(?:image_)?\d+(?:x|_)\d+_", re.IGNORECASE)


def strip_size_prefix(filename):
    return SIZE_PREFIX_RE.sub("", filename)


def candidate_keys_local(old_path):
    if old_path is None or pd.isna(old_path):
        return

    parts = [
        p for p in PurePosixPath(str(old_path)).parts
        if p not in ("", "/")
    ]

    if not parts:
        return

    filename = parts[-1]
    filenames = [filename]

    stripped = strip_size_prefix(filename)
    if stripped != filename:
        filenames.append(stripped)

    for fname in filenames:
        fixed_parts = parts[:-1] + [fname]
        max_n = min(MAX_TAIL_PARTS, len(fixed_parts))

        for n in range(max_n, 0, -1):
            yield tail_index, n, "/".join(fixed_parts[-n:]).lower()

            stem_parts = list(fixed_parts[-n:])
            stem_parts[-1] = os.path.splitext(stem_parts[-1])[0]
            yield stem_tail_index, n, "/".join(stem_parts).lower()


def find_new_image_path_local(old_path):
    for index, n, key in candidate_keys_local(old_path) or []:
        if index is None:
            continue

        hit = index[n].get(key)
        if hit and hit != AMBIG:
            return hit

    return pd.NA

In [47]:
# 1. Collect unique old image paths
all_old_paths = set()

for paths in tqdm(df_2_["images"], desc="Collecting unique image paths"):
    if isinstance(paths, list):
        all_old_paths.update(paths)

print("unique old image paths:", len(all_old_paths))

# 2. Map each unique path once
old_to_new = {
    p: find_new_image_path_local(p)
    for p in tqdm(all_old_paths, desc="Mapping unique image paths")
}

# 3. Rebuild list column
def map_image_list(paths):
    if not isinstance(paths, list):
        return pd.NA

    return [old_to_new.get(p, pd.NA) for p in paths]


df_2_["new_images"] = df_2_["images"].progress_map(map_image_list)

unique old image paths: 1055751


Mapping unique image paths:   0%|          | 0/1055751 [00:00<?, ?it/s]

  0%|          | 0/1055751 [00:00<?, ?it/s]

In [48]:
df_2_

,images,messages,new_images
0,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate a detail...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
1,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate a medica...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
2,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate a medica...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
3,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Impr...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
4,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Find...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
...,...,...,...
1055746,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Summ...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
1055747,[/raid1/akshay/mimic_data/physionet.org/files/...,"[{'from': 'human', 'value': 'Generate the Impr...",[/fastdata3/cxr_data/opensource_datasets/mimic...
1055748,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Summ...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
1055749,[/raid/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the Prom...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....


In [49]:
total_imgs = df_2_["new_images"].map(lambda x: len(x) if isinstance(x, list) else 0).sum()

mapped_imgs = df_2_["new_images"].map(
    lambda x: sum(pd.notna(p) for p in x) if isinstance(x, list) else 0
).sum()

print("rows:", len(df_2_))
print("total images:", total_imgs)
print("mapped images:", mapped_imgs)
print("missing images:", total_imgs - mapped_imgs)

rows: 1055751
total images: 1055751
mapped images: 1055549
missing images: 202


In [50]:
df_2_ = df_2_[df_2_["new_images"].map(lambda x: isinstance(x, list) and all(pd.notna(p) for p in x))]
df_2_ = df_2_.drop(columns=["images"]).rename(columns={"new_images": "images"})
df_2_

,messages,images
0,"[{'from': 'human', 'value': 'Generate a detail...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
1,"[{'from': 'human', 'value': 'Generate a medica...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
2,"[{'from': 'human', 'value': 'Generate a medica...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
3,"[{'from': 'human', 'value': 'Generate the Impr...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
4,"[{'from': 'human', 'value': 'Generate the Find...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
...,...,...
1055746,"[{'from': 'human', 'value': 'Generate the Summ...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
1055747,"[{'from': 'human', 'value': 'Generate the Impr...",[/fastdata3/cxr_data/opensource_datasets/mimic...
1055748,"[{'from': 'human', 'value': 'Generate the Summ...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....
1055749,"[{'from': 'human', 'value': 'Generate the Prom...",[/fastdata3/cxr_data/segmed_data/images/1.3.6....


In [52]:
import os
from tqdm.auto import tqdm

missing = []

for paths in tqdm(df_2_["images"], desc="Verifying images on disk"):
    for p in paths:
        if not os.path.exists(p):
            missing.append(p)

missing = sorted(set(missing))

print("missing:", len(missing))
print(missing[:20])

assert len(missing) == 0, f"{len(missing)} images are missing"

Verifying images on disk:   0%|          | 0/1055549 [00:00<?, ?it/s]

missing: 0
[]


In [53]:
df_2_.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/15_July_final_segmed_chexpert_mimic_vlm_conv_gemini_response_neysa.json", lines=True, orient="records")

# data 3

In [54]:
df_3_ = pd.read_json("/fastdata3/vlm_data_parquets/11_Nov_pretraining_data_without_bbox.json", lines=True)

In [55]:
df_3_

,image,messages
0,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Give a detailed r..."
1,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Give a detailed r..."
2,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Give a detailed r..."
3,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Give a detailed r..."
4,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Give a detailed r..."
...,...,...
336243,/raid3/opensource_datasets/PNG/pngs/PNG/train/...,"[{'from': 'human', 'value': 'Do the detailed a..."
336244,/raid3/opensource_datasets/PNG/pngs/PNG/train/...,"[{'from': 'human', 'value': 'Do the detailed a..."
336245,/raid3/opensource_datasets/PNG/pngs/PNG/train/...,"[{'from': 'human', 'value': 'Do the detailed a..."
336246,/raid3/opensource_datasets/PNG/pngs/PNG/train/...,"[{'from': 'human', 'value': 'Do the detailed a..."


In [56]:
from tqdm.auto import tqdm
import pandas as pd

tqdm.pandas(desc="Mapping df_3_ image paths")

df_3_["new_image"] = df_3_["image"].progress_map(find_new_image_path_local)

print("rows:", len(df_3_))
print("mapped:", df_3_["new_image"].notna().sum())
print("missing:", df_3_["new_image"].isna().sum())

Mapping df_3_ image paths:   0%|          | 0/336248 [00:00<?, ?it/s]

rows: 336248
mapped: 306610
missing: 29638


In [59]:
df_3_ = df_3_[df_3_["new_image"].notna()].copy()

df_3_ = (
    df_3_
    .drop(columns=["image"])
    .rename(columns={"new_image": "image"})
    .reset_index(drop=True)
)

print(df_3_.shape)

(306610, 2)


In [92]:
df_3_.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/11_Nov_pretraining_data_without_bbox_neysa.json", lines=True, orient="records")

# df 4

In [157]:
df_4_ = pd.read_json("/fastdata3/vlm_data_parquets/14_Nov_corrected_internvl_1000res_cxr_nodule_bbox_data_with_conversation_and_target_res.json", lines=True)

In [158]:
df_4_

,image,conversation
0,/raid3/cxr_data/training/images/max.dev4.16473...,"[{'from': 'human', 'value': 'Generate the bbox..."
1,/raid3/cxr_data/training/images/fiveceu.5ee44f...,"[{'from': 'human', 'value': 'Generate the bbox..."
2,/raid3/cxr_data/training/images/bodyscans.1.2....,"[{'from': 'human', 'value': 'Generate the bbox..."
3,/raid3/cxr_data/training/images/medanta.dd127b...,"[{'from': 'human', 'value': 'Generate the bbox..."
4,/raid3/cxr_data/training/images/fiveceu.989adf...,"[{'from': 'human', 'value': 'Generate the bbox..."
...,...,...
618001,/raid2/cxr_data/training/images/image_1288x128...,"[{'from': 'human', 'value': 'Generate the bbox..."
618002,/raid2/cxr_data/training/images/image_1204x120...,"[{'from': 'human', 'value': 'Generate the bbox..."
618003,/raid2/cxr_data/training/images/image_1456x145...,"[{'from': 'human', 'value': 'Generate the bbox..."
618004,/raid2/cxr_data/training/images/image_1540x154...,"[{'from': 'human', 'value': 'Generate the bbox..."


In [159]:
df_4_["_new_image"] = df_4_["image"].progress_map(find_new_image_path_keep_resolution_name)

valid_mask = df_4_["_new_image"].notna()

print("rows before:", len(df_4_))
print("valid rows:", valid_mask.sum())
print("dropped rows:", (~valid_mask).sum())

df_4_bad = df_4_[~valid_mask].copy()
df_4_ = df_4_[valid_mask].copy()

df_4_ = (
    df_4_
    .drop(columns=["image"])
    .rename(columns={"_new_image": "image"})
    .reset_index(drop=True)
)

print(df_4_.shape)

  0%|          | 0/618006 [00:00<?, ?it/s]

rows before: 618006
valid rows: 618006
dropped rows: 0
(618006, 2)


In [168]:
df_4_.image.iloc[618003]

'/localstorage/cxr_data/internal_data/training/images/images/image_1456x1456_ca.phase4.unit9.4.d3c7b2d80e2edeeeff077bcda7b4b4a70ec62ab932ba353de16dd9d7.x.2.png'

In [169]:
df_4_.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/14_Nov_corrected_internvl_1000res_cxr_nodule_bbox_data_with_conversation_and_target_res.json", orient="records", lines=True)

# data 5

In [102]:
df_5 = pd.read_json("/fastdata3/vlm_data_parquets/13_Nov_data_with_ASP_and_multilevel_reasoning.json", lines=True)

In [106]:
df_5

,image,messages
0,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Generate Answer S..."
1,/raid/segmed_data/segmed_images/1.3.6.1.4.1.55...,"[{'from': 'human', 'value': 'Generate Answer S..."
2,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Generate Answer S..."
3,/raid/segmed_data/segmed_images/1.3.6.1.4.1.55...,"[{'from': 'human', 'value': 'Generate Answer S..."
4,/raid/segmed_data/segmed_images/1.3.6.1.4.1.55...,"[{'from': 'human', 'value': 'Generate Answer S..."
...,...,...
710861,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Generate Answer S..."
710862,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Generate Answer S..."
710863,/raid/segmed_data/segmed_images/1.3.6.1.4.1.55...,"[{'from': 'human', 'value': 'Generate Answer S..."
710864,/raid/opensource_datasets/PNG/pngs/PNG/train/p...,"[{'from': 'human', 'value': 'Generate Answer S..."


In [107]:
from tqdm.auto import tqdm
import pandas as pd

tqdm.pandas(desc="Mapping df_5_ image paths")

# Map old image paths to new server image paths
df_5["new_image"] = df_5["image"].progress_map(find_new_image_path_local)

print("rows:", len(df_5))
print("mapped:", df_5["new_image"].notna().sum())
print("missing:", df_5["new_image"].isna().sum())

# Drop unmapped rows, replace image column
df_5_ = df_5[df_5["new_image"].notna()].copy()

df_5 = (
    df_5
    .drop(columns=["image"])
    .rename(columns={"new_image": "image"})
    .reset_index(drop=True)
)

print(df_5.shape)

Mapping df_5_ image paths:   0%|          | 0/710866 [00:00<?, ?it/s]

rows: 710866
mapped: 710650
missing: 216
(710866, 2)


In [108]:
df_5.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/13_Nov_data_with_ASP_and_multilevel_reasoning_neysa.json", lines=True, orient="records")

# df 6

In [119]:
df_6 = pd.read_json("/fastdata3/vlm_data_parquets/19_Nov_training_data_internvl_1000res_bbox_data_monochrome2_segmed_with_png_res.json", lines=True)
df_6

,image,conversation
0,/raid3/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the bbox..."
1,/raid3/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the bbox..."
2,/raid3/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the bbox..."
3,/raid3/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the bbox..."
4,/raid3/segmed_data/segmed_images/1.3.6.1.4.1.5...,"[{'from': 'human', 'value': 'Generate the bbox..."
...,...,...
1099475,/raid27/sahil_work/prod_data/us-qureapp-dicom-...,"[{'from': 'human', 'value': 'Generate the bbox..."
1099476,/raid27/sahil_work/prod_data/us-qureapp-dicom-...,"[{'from': 'human', 'value': 'Generate the bbox..."
1099477,/raid27/sahil_work/prod_data/us-qureapp-dicom-...,"[{'from': 'human', 'value': 'Generate the bbox..."
1099478,/raid27/sahil_work/prod_data/us-qureapp-dicom-...,"[{'from': 'human', 'value': 'Generate the bbox..."


In [120]:
from tqdm.auto import tqdm
import pandas as pd

tqdm.pandas(desc="Mapping df_5_ image paths")
from tqdm.auto import tqdm


def map_image_paths(df, path_mapper_func):
    """
    Maps old image paths to new server image paths for any dataframe with 'image' column using the given path_mapper_func.
    Drops unmapped rows (where the new image path is NA) and replaces 'image' with 'new_image'.
    Returns the processed dataframe and the filtered dataframe with only mapped rows.
    """
    
    tqdm.pandas(desc="Mapping image paths")
    df["new_image"] = df["image"].progress_map(path_mapper_func)

    print("rows:", len(df))
    print("mapped:", df["new_image"].notna().sum())
    print("missing:", df["new_image"].isna().sum())

    # DataFrame with only mapped rows
    df_mapped = df[df["new_image"].notna()].copy()

    # Replace image column, drop old one, reset index
    df_processed = (
        df
        .drop(columns=["image"])
        .rename(columns={"new_image": "image"})
        .reset_index(drop=True)
    )
    return df_processed

# Example usage:
df_6 = map_image_paths(df_6, find_new_image_path_local)

print(df_6.shape)

Mapping image paths:   0%|          | 0/1099480 [00:00<?, ?it/s]

rows: 1099480
mapped: 1099480
missing: 0
(1099480, 2)


In [122]:
df_6.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/19_Nov_training_data_internvl_1000res_bbox_data_monochrome2_segmed_with_png_res_neysa.json", lines=True, orient="records")

# data 7

In [123]:
df_7 = pd.read_json("/fastdata3/vlm_data_parquets/08_June_2026_final_report_comparison_data_deepseek_490k.json", lines=True)
df_7

In [124]:
df_7

,images,conversations
0,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
1,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
2,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
3,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
4,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
...,...,...
490990,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
490991,[/raid3/opensource_datasets/PNG/pngs/PNG/train...,"[{'from': 'human', 'value': 'Compare the radio..."
490992,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."
490993,[/raid3/segmed_data/segmed_images/1.3.6.1.4.1....,"[{'from': 'human', 'value': 'Compare the radio..."


In [129]:
df_7.conversations.iloc[1], df_7.images.iloc[1]

([{'from': 'human',
   'value': 'Compare the radiographic findings across the images. Analyze each image individually, identify key similarities and differences, infer relative disease severity, and summarize the comparison using a structured knowledge graph. Return the final output as JSON. '},
  {'from': 'gpt',
   'value': '**SECTION 1: Detailed Chain-of-Thought Analysis and Verbalized Knowledge Graph**\n\n**A) Step-by-Step Chain-of-Thought (CoT) Analysis**\n\nI am performing a side-by-side visual comparison of two chest radiographs, systematically searching anatomical regions.\n\n**Step 1: Support Devices and Patient Positioning**\n- Image 1: No endotracheal tubes, central venous catheters, or other support devices are visible. The patient is upright with satisfactory inspiratory effort given the hyperinflated lungs.\n- Image 2: No support devices are visible. Patient positioning appears similar, upright.\n- Visual Difference: No difference; both radiographs are free of support devi

In [131]:
from tqdm.auto import tqdm
import pandas as pd

tqdm.pandas(desc="Mapping df_7 image lists")

# Collect unique old paths once
all_old_paths = set()

for paths in tqdm(df_7["images"], desc="Collecting unique image paths"):
    if isinstance(paths, list):
        all_old_paths.update(paths)

print("unique old image paths:", len(all_old_paths))

# Map each unique path once
old_to_new = {
    p: find_new_image_path_local(p)
    for p in tqdm(all_old_paths, desc="Mapping unique image paths")
}


def map_image_list_fast(paths):
    if not isinstance(paths, list):
        return pd.NA

    return [old_to_new.get(p, pd.NA) for p in paths]


df_7["new_image"] = df_7["images"].progress_map(map_image_list_fast)

# Drop rows where any image failed to map
valid_mask = df_7["new_image"].map(
    lambda x: isinstance(x, list) and len(x) > 0 and all(pd.notna(p) for p in x)
)

print("rows:", len(df_7))
print("valid rows:", valid_mask.sum())
print("dropped rows:", (~valid_mask).sum())

df_7 = df_7[valid_mask].copy()

# Replace image column
df_7 = (
    df_7
    .drop(columns=["images"])
    .rename(columns={"new_image": "images"})
    .reset_index(drop=True)
)

print(df_7.shape)

unique old image paths: 507376


Mapping unique image paths:   0%|          | 0/507376 [00:00<?, ?it/s]

Mapping df_7 image lists:   0%|          | 0/490995 [00:00<?, ?it/s]

rows: 490995
valid rows: 490796
dropped rows: 199
(490796, 2)


In [133]:
df_7.images.iloc[1]

['/fastdata3/cxr_data/segmed_data/images/1.3.6.1.4.1.55648.040649988397901062111098857723974046499.1.1.green.png',
 '/fastdata3/cxr_data/opensource_datasets/mimic/images/p10/p10293407/s51336475/23076b0c-400e0d94-d5748866-807da546-4860ba97.jpg']

In [135]:
df_7.conversations.iloc[1]

[{'from': 'human',
  'value': 'Compare the radiographic findings across the images. Analyze each image individually, identify key similarities and differences, infer relative disease severity, and summarize the comparison using a structured knowledge graph. Return the final output as JSON. '},
 {'from': 'gpt',
  'value': '**SECTION 1: Detailed Chain-of-Thought Analysis and Verbalized Knowledge Graph**\n\n**A) Step-by-Step Chain-of-Thought (CoT) Analysis**\n\nI am performing a side-by-side visual comparison of two chest radiographs, systematically searching anatomical regions.\n\n**Step 1: Support Devices and Patient Positioning**\n- Image 1: No endotracheal tubes, central venous catheters, or other support devices are visible. The patient is upright with satisfactory inspiratory effort given the hyperinflated lungs.\n- Image 2: No support devices are visible. Patient positioning appears similar, upright.\n- Visual Difference: No difference; both radiographs are free of support devices.

In [136]:
df_7.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/08_June_2026_final_report_comparison_data_deepseek_490k_neysa.json", lines=True, orient="records")

# data 8

In [149]:
df_8 = pd.read_json("/fastdata3/vlm_data_parquets/05_June_2026_vlm_bbox_comparison_data_550k.json", lines=True)

In [150]:
df_8

,images,conversations
0,[/raid3/cxr_data/training/images/image_756x756...,"[{'from': 'human', 'value': 'Generate the boun..."
1,[/raid3/cxr_data/training/images/image_1036x10...,"[{'from': 'human', 'value': 'Generate the boun..."
2,[/raid3/cxr_data/training/images/image_1288x12...,"[{'from': 'human', 'value': 'Generate the boun..."
3,[/raid3/cxr_data/training/images/image_840x840...,"[{'from': 'human', 'value': 'Generate the boun..."
4,[/raid3/cxr_data/training/images/medall.1.2.84...,"[{'from': 'human', 'value': 'Generate the boun..."
...,...,...
562614,[/raid2/cxr_data/training/images/image_1260x12...,"[{'from': 'human', 'value': 'Generate the boun..."
562615,[/raid2/cxr_data/training/images/medanta.a25d3...,"[{'from': 'human', 'value': 'Generate the boun..."
562616,[/raid2/cxr_data/training/images/fiveceu.93e40...,"[{'from': 'human', 'value': 'Generate the boun..."
562617,[/raid2/cxr_data/training/images/fiveceu.dcfe6...,"[{'from': 'human', 'value': 'Generate the boun..."


In [151]:
import os
import re
import pandas as pd
from tqdm.auto import tqdm

tqdm.pandas()

RES_PREFIX_RE = re.compile(r"^(?:image_)?\d+(?:x|_)\d+_", re.IGNORECASE)


def find_new_image_path_keep_resolution_name(old_path):
    new_path = find_new_image_path_local(old_path)

    if pd.isna(new_path):
        return pd.NA

    old_fname = os.path.basename(str(old_path))

    # image_756x756_abc.png -> /new/dir/image_756x756_abc.png
    if RES_PREFIX_RE.match(old_fname):
        return os.path.join(os.path.dirname(str(new_path)), old_fname)

    return new_path

In [152]:
all_old_paths = set()

for paths in tqdm(df_8["images"], desc="Collecting unique df_8 image paths"):
    if isinstance(paths, list):
        all_old_paths.update(paths)
    else:
        raise TypeError(f"Non-list value found: {type(paths)}")

old_to_new = {
    p: find_new_image_path_keep_resolution_name(p)
    for p in tqdm(all_old_paths, desc="Mapping with resolution filenames")
}


def map_image_list_keep_resolution(paths):
    return [old_to_new.get(p, pd.NA) for p in paths]


df_8["_new_images"] = df_8["images"].progress_map(map_image_list_keep_resolution)

valid_mask = df_8["_new_images"].map(
    lambda x: isinstance(x, list) and len(x) > 0 and all(pd.notna(p) for p in x)
)

print("rows before:", len(df_8))
print("valid rows:", valid_mask.sum())
print("dropped rows:", (~valid_mask).sum())

df_8 = df_8[valid_mask].copy()

df_8 = (
    df_8
    .drop(columns=["images"])
    .rename(columns={"_new_images": "images"})
    .reset_index(drop=True)
)

print(df_8.shape)

Mapping with resolution filenames:   0%|          | 0/435456 [00:00<?, ?it/s]

  0%|          | 0/562619 [00:00<?, ?it/s]

rows before: 562619
valid rows: 562619
dropped rows: 0
(562619, 2)


In [156]:
df_8.to_json("/localstorage/sahil_work/VLM_qure/data_preparation/training_data/05_June_2026_vlm_bbox_comparison_data_550k_neysa.json", lines=True, orient="records")

# data 7

In [171]:
df = pd.read_json("/fastdata3/vlm_data_parquets//raid4/janhavi/vlm_data/05_June_2026_vlm_bbox_label_matching_data_617k.json", lines=True)

FileNotFoundError: File /fastdata3/vlm_data_parquets//raid4/janhavi/vlm_data/05_June_2026_vlm_bbox_label_matching_data_617k.json does not exist